# Stage 1b: Calculate Physical Indices

This notebook calculates derived physical properties for the entire dataset and updates `merged_gdf.feather`.

Specifically, it computes:
1. **Potential Density Anomaly ($\sigma_0$)**: Using the Gibbs SeaWater (GSW) Oceanographic Toolbox.
2. **Stratification Indices**:
   - **Density Difference** ($\Delta ho = ho_{bottom} - ho_{surface}$)
   - **Density Gradient** ($\Delta ho / \Delta z = (ho_{bottom} - ho_{surface}) / (z_{bottom} - z_{surface})$)

By calculating these here, they are permanently stored in the main dataset and become immediately available for all downstream analysis (seasonal subsets, PCA, environmental correlations) and dashboard visualization.


In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
import gsw
import sys
from pathlib import Path

# Add project root to sys.path
project_root = Path("../../").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from utils.paths import ProjectPaths

print(f"Project root: {project_root}")


Project root: /home


## 1. Load the Processed Dataset
We load `merged_gdf.feather` which was produced by `01_Data_Processing.ipynb`.


In [2]:
feather_path = ProjectPaths.MERGED_DATA_FILE
print(f"Loading data from {feather_path}...")
merged_gdf = gpd.read_feather(feather_path)

print(f"Loaded {len(merged_gdf):,} records.")


Loading data from /home/reyha/SIS_Analysis_WSL/data/processed/merged_gdf.feather...
Loaded 62,738 records.


## 2. Calculate Potential Density ($\sigma_0$)
Density is calculated row-by-row. We require valid practical salinity (`salt2` or `salt`), in-situ temperature (`wtemp`), and coordinates (`latitude`, `longitude`).


In [3]:
def compute_density_row_wise(df):
    """
    Compute Potential Density Anomaly (sigma0) for all rows with valid T and S.
    Uses 'salt2' primarily, falling back to 'salt' if needed.
    """
    # Create a copy to avoid SettingWithCopyWarning if working on slices
    result = pd.Series(index=df.index, dtype=float)
    
    # Identify salinity column to use (prefer 'salt' as it is the mapped column from V2)
    salt_col = 'salt' if 'salt' in df.columns else 'salt2'
    
    # Filter valid rows
    valid_mask = df['wtemp'].notna() & df[salt_col].notna() & df['latitude'].notna() & df['longitude'].notna()
    valid_df = df[valid_mask]
    
    if len(valid_df) == 0:
        return result
        
    print(f"Calculating density for {len(valid_df):,} valid rows...")
    
    # Convert to numpy arrays for fast GSW vectorised calculation
    SP = valid_df[salt_col].values
    t = valid_df['wtemp'].values
    lon = valid_df['longitude'].values
    lat = valid_df['latitude'].values
    
    # GSW steps
    SA = gsw.SA_from_SP(SP, 0, lon, lat)
    CT = gsw.CT_from_t(SA, t, 0)
    sigma0 = gsw.sigma0(SA, CT)
    
    # Assign back to series
    result.loc[valid_mask] = sigma0
    return result

merged_gdf['density'] = compute_density_row_wise(merged_gdf)
print(f"Computed density for {merged_gdf['density'].notna().sum():,} records.")


Calculating density for 62,651 valid rows...
Computed density for 62,651 records.


## 3. Calculate Stratification Indices
Stratification requires pairing a surface measurement (`saisuipoin == 1`) with a bottom measurement (`saisuipoin == 2`) taken at the same station on the same day.

We will:
1. Extract surface and bottom subsets.
2. Merge them on `['zettaicode', 'survey_ym', 'surveyday']`.
3. Calculate the difference and gradient.
4. Map the results back to the main dataframe (we will assign the stratification value to **both** the surface and bottom rows of that pair for convenience).


In [4]:
print("Pairing surface and bottom measurements...")

# 1. Subset surface and bottom, keeping necessary columns
key_cols = ['zettaicode', 'survey_ym', 'surveyday']
val_cols = ['density', 'saisuidept']

surf_df = merged_gdf[merged_gdf['saisuipoin'] == 1][key_cols + val_cols].copy()
bot_df = merged_gdf[merged_gdf['saisuipoin'] == 2][key_cols + val_cols].copy()

# Rename value columns to distinguish
surf_df = surf_df.rename(columns={'density': 'density_surf', 'saisuidept': 'depth_surf'})
bot_df = bot_df.rename(columns={'density': 'density_bot', 'saisuidept': 'depth_bot'})

# Drop duplicates just in case there are multiple samples at the exact same location/day/layer
surf_df = surf_df.drop_duplicates(subset=key_cols)
bot_df = bot_df.drop_duplicates(subset=key_cols)

# 2. Merge
paired_df = pd.merge(surf_df, bot_df, on=key_cols, how='inner')
print(f"Found {len(paired_df):,} matching surface-bottom pairs.")

# Ensure depth is numeric
paired_df['depth_surf'] = pd.to_numeric(paired_df['depth_surf'], errors='coerce')
paired_df['depth_bot'] = pd.to_numeric(paired_df['depth_bot'], errors='coerce')

# 3. Calculate indices
# Density Difference
paired_df['stratification'] = paired_df['density_bot'] - paired_df['density_surf']

# Density Gradient
depth_diff = paired_df['depth_bot'] - paired_df['depth_surf']
# Avoid division by zero
depth_diff = depth_diff.replace(0, np.nan)
paired_df['stratification_gradient'] = paired_df['stratification'] / depth_diff

# 4. Map back to main dataframe
# We create a mapping index from the keys
merged_gdf = merged_gdf.drop(columns=['stratification', 'stratification_gradient'], errors='ignore')

# Merge the indices back via a left join
# This will broadcast the stratification values to all rows that match the key (both surface and bottom)
merged_gdf = pd.merge(
    merged_gdf,
    paired_df[key_cols + ['stratification', 'stratification_gradient']],
    on=key_cols,
    how='left'
)

print(f"Assigned 'stratification' to {merged_gdf['stratification'].notna().sum():,} records.")
print(f"Assigned 'stratification_gradient' to {merged_gdf['stratification_gradient'].notna().sum():,} records.")


Pairing surface and bottom measurements...
Found 31,305 matching surface-bottom pairs.
Assigned 'stratification' to 62,578 records.
Assigned 'stratification_gradient' to 60,580 records.


## 4. Inspect the Updates
Let's look at a sample station to verify the calculations.


In [5]:
# Find a station with valid stratification
sample_station = merged_gdf[merged_gdf['stratification'].notna()]['zettaicode'].iloc[0]

cols_to_view = ['zettaicode', 'surveyyear', 'surveymont', 'saisuipoin', 'saisuidept', 'wtemp', 'salt', 'density', 'stratification', 'stratification_gradient']
display(merged_gdf[merged_gdf['zettaicode'] == sample_station][cols_to_view].head(10))


,zettaicode,surveyyear,surveymont,saisuipoin,saisuidept,wtemp,salt,density,stratification,stratification_gradient
0,00002353,1982,1,1,0.5,6.3,0.0,-0.066041,-0.024152,-0.004391
1,00002353,1981,5,2,7.0,15.1,0.0,-0.911963,0.190675,0.029335
2,00002353,1981,7,2,8.0,21.6,0.0,-2.135955,2.274655,0.303287
66,00002353,1981,5,1,0.5,16.3,0.0,-1.102638,0.190675,0.029335
67,00002353,1981,7,1,0.5,30.2,0.0,-4.410610,2.274655,0.303287
68,00002353,1981,10,1,0.5,19.1,0.0,-1.610833,0.000000,0.000000
76,00002353,1981,10,2,7.0,19.1,0.0,-1.610833,0.000000,0.000000
77,00002353,1982,1,2,6.0,6.9,0.0,-0.090193,-0.024152,-0.004391
1478,00002353,1982,7,1,0.5,23.0,0.0,-2.458076,0.047220,0.006296
1538,00002353,1982,10,1,0.5,18.4,0.0,-1.475682,0.037636,0.005018


## 5. Save the Updated Dataset
Overwriting `merged_gdf.feather` with the new columns.


In [6]:
print(f"Saving updated data back to {feather_path}...")
merged_gdf.to_feather(feather_path)
print("Save complete.")


Saving updated data back to /home/reyha/SIS_Analysis_WSL/data/processed/merged_gdf.feather...


Save complete.
